In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
from functools import reduce
from collections import Counter
import os
import re
import warnings
warnings.filterwarnings('ignore')
pd.options.display.max_columns = None
from IPython.display import display

In [ ]:
file_path = "../data/processed/batch2019_first_year_data.csv"

# Read the CSV file, treating the first column as index and skipping it
first_year_data = pd.read_csv(file_path)
# Drop any 'Unnamed' columns
first_year_data = first_year_data.loc[:, ~first_year_data.columns.str.contains('^Unnamed')]
first_year_data

# More feature Engineering 

In [ ]:
# create a new column called EXAM_REGIS_WINDOW in the first_year_data DataFrame, where the values are the 
# difference between the columns DATE_EXAM_CONDUCTED and EXAM_REGISTRATION_DATE in weeks


# Ensure both columns are in datetime format
first_year_data['DATE_EXAM_CONDUCTED'] = pd.to_datetime(first_year_data['DATE_EXAM_CONDUCTED'], errors='coerce')
first_year_data['EXAM_REGISTRATION_DATE'] = pd.to_datetime(first_year_data['EXAM_REGISTRATION_DATE'], errors='coerce')

# Calculate the difference between START_DATE and DATE_ENROLLMENT_REQUEST in weeks
first_year_data['EXAM_REGIS_WINDOW'] = (first_year_data['DATE_EXAM_CONDUCTED'] - first_year_data['EXAM_REGISTRATION_DATE']).dt.total_seconds() / (7 * 24 * 60 * 60)

# Round the values to the nearest integer
first_year_data['EXAM_REGIS_WINDOW'] = first_year_data['EXAM_REGIS_WINDOW'].round().astype(int)

# Delete rows with negative ENROLLMENT_WINDOW
first_year_data = first_year_data[first_year_data['EXAM_REGIS_WINDOW'] >= 0]

# Display the cleaned DataFrame
first_year_data[['DATE_EXAM_CONDUCTED', 'EXAM_REGISTRATION_DATE', 'EXAM_REGIS_WINDOW']]

In [ ]:
first_year_data['EXAM_REGIS_WINDOW'].value_counts()

In [ ]:
first_year_data.head(3)

# Delete irrelevant columns for each year data eg a radom code

In [ ]:
# List of columns to delete
columns_to_delete = ['ACADEMIC_YEAR',
                     'START_DATE', 
                     'END_DATE', 
                     'PROGRAM_ACTIVE_CODE', 
                     'DATE_ACTIVE_CODE_MODIFIED', 
                     'HIGHER_EDU_REGISTER', 
                     'DATE_REGIS_PROOF',
                     'DATE_ENROLLMENT_REQUEST',
                     'DATE_EXAM_STARTED',
                     'EXAM_PROG_DETAILS',
                     'DATE_EXAM_CONDUCTED',
                     'EXAM_REGISTRATION_DATE',
                     'DATE_EXAM_DETAILS_MODIFIED',
                     'DATE_ACTIVE_CODE_MODIFIED', 
                     'PRIOR_EDUCATION', 
                     'SCHOOL_DESCRIPTION', 
                     'PRIOR_FINAL_EXAM_DATE', 
                     'PRIOR_VERIFICATION_DATE',
                     'BIRTH_DATE',
                     'YEAR_STARTED_HU',
                     'YEAR',
                     'EXAM_REGIS_WINDOW'] 

# Delete these columns from the DataFrame
first_year_data = first_year_data.drop(columns=columns_to_delete)

# Display the DataFrame after deleting the columns
first_year_data

In [ ]:
first_year_data['DROPOUT_RISK'].value_counts()

# Group

In [ ]:
first_year_data.info()

In [ ]:
first_year_data['PRIOR_EXAM_YEAR'] = first_year_data['PRIOR_EXAM_YEAR'].astype(str)
first_year_data['DROPOUT_RISK'] = first_year_data['DROPOUT_RISK'].astype(str)

In [ ]:
#Group by STUDENT_NUM and handle both object columns (like strings) with ' '.join and 
# numeric columns (like integers or floats) by calculating their average

In [ ]:
# Define the aggregation functions: ' '.join for object columns and 'mean' for numeric columns
aggregation_functions = {
    col: (' '.join if first_year_data[col].dtype == 'O' else 'mean') 
    for col in first_year_data.columns if col != 'STUDENT_NUM'
}

# Ensure 'STUDENT_NUM' is treated as a string
first_year_data['STUDENT_NUM'] = first_year_data['STUDENT_NUM'].astype(str)

# Apply groupby and aggregation using the defined functions
grouped_first_year_data = first_year_data.groupby('STUDENT_NUM').agg(aggregation_functions).reset_index()

grouped_first_year_data


In [ ]:
# Function to return the most frequent string in a list or the unique one
def get_most_frequent_or_unique(strings):
    # Check if all strings are the same
    if len(set(strings)) == 1:
        return strings[0]  # Return the single unique string
    else:
        # If there are different strings, return the most common one
        return Counter(strings).most_common(1)[0][0]

# Iterate through columns in the DataFrame
for col in grouped_first_year_data.select_dtypes(include='object').columns:
    # Apply the function across each row to find most frequent or unique string
    grouped_first_year_data[col] = grouped_first_year_data[col].apply(lambda x: get_most_frequent_or_unique(x.split()) if isinstance(x, str) else x)

grouped_first_year_data


In [ ]:
grouped_first_year_data['DROPOUT_RISK'].value_counts()

In [ ]:
# Print value counts for all columns
for col in grouped_first_year_data.columns:
    print(f"Value counts for column: {col}")
    print(grouped_first_year_data[col].value_counts(dropna=False))  # include NaN values in the count
    print("\n" + "="*50 + "\n")

In [ ]:
# Function to convert object columns with two unique values into binary
def convert_to_binary(col):
    unique_vals = col.unique()
    
    # Check if the column has exactly two unique values
    if len(unique_vals) == 2:
        # Check if one of the values is 'J' and assign 1 to 'J', 0 to the other
        if 'J' in unique_vals:
            col = col.map({unique_vals[0]: 1, unique_vals[1]: 0}) if unique_vals[0] == 'J' else col.map({unique_vals[1]: 1, unique_vals[0]: 0})
        else:
            # Assign 1 to one of the values and 0 to the other
            col = col.map({unique_vals[0]: 1, unique_vals[1]: 0})
    
    return col

# Apply the conversion to all object-type columns in grouped_first_year_data
for col in grouped_first_year_data.select_dtypes(include='object').columns:
    grouped_first_year_data[col] = convert_to_binary(grouped_first_year_data[col])

# Display the modified DataFrame 
grouped_first_year_data

In [ ]:
# Find and delete columns where all values are the same
columns_to_delete = [col for col in grouped_first_year_data.columns if grouped_first_year_data[col].nunique() == 1]

# Drop these columns from the DataFrame
grouped_first_year_data = grouped_first_year_data.drop(columns=columns_to_delete)

# Display the updated DataFrame and the removed columns
print(f"Deleted columns: {columns_to_delete}")
grouped_first_year_data


In [ ]:
grouped_first_year_data.info()

In [ ]:
grouped_first_year_data['STUDENT_NUM'] = grouped_first_year_data['STUDENT_NUM'].astype(int)

In [ ]:
encoded_data = grouped_first_year_data

In [ ]:
# Identify object-type columns
object_columns = encoded_data.select_dtypes(include='object').columns

# Perform one-hot encoding
encoded_data = pd.get_dummies(encoded_data, columns=object_columns, prefix=object_columns, drop_first=False)

# Loop through columns and check for True/False values and convert them to 1/0
for col in encoded_data.columns:
    if encoded_data[col].dtype == 'bool':
        encoded_data[col] = encoded_data[col].astype(int)

# Display the updated DataFrame with 1s and 0s
encoded_data

In [ ]:
# Move the DROPOUT_RISK column to the rightmost position
dropout_risk_col = encoded_data.pop('DROPOUT_RISK')

# Add it back as the last column
encoded_data['DROPOUT_RISK'] = dropout_risk_col

encoded_data
